# ML surrogates for a 1T′-WTe₂ topological-insulator FET
**ECE 455/555 course project, Shuvankar Biswas**  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKBiswas1998/ML-Project-for-ECE/blob/test_1/notebooks/WTe2_TIFET_pipeline.ipynb)

This notebook runs the whole pipeline from the GitHub repository
[SKBiswas1998/ML-Project-for-ECE](https://github.com/SKBiswas1998/ML-Project-for-ECE) on Google Colab or Kaggle (CPU is enough).
All data come from the repository: 798 NEGF gate sweeps (38 temperatures × 21 edge-vacancy levels, 20 gate points each).

Every cell only **calls functions from the `wte2ml` package** and shows the result. The logic lives in `python/wte2ml/`, and design choices are in `docs/DECISIONS.md`.

**Run time.** With the default settings (`RERUN_CLASSICAL = RERUN_MLP = False`) the notebook reads the committed model results and finishes in about 2 minutes. Setting a flag to `True` recomputes those models from scratch. On an ordinary laptop CPU that took about 31 min for the classical models and about 100 min for the MLP (serial, all 8 targets).

## 0. Settings: which branch, and should the slow model runs be repeated?

In [ ]:
REPO_URL = "https://github.com/SKBiswas1998/ML-Project-for-ECE.git"
BRANCH = "test_1"          # branch with the Python pipeline
RERUN_CLASSICAL = False    # True: refit baselines, ridge, boosting, GP (~31 min on CPU)
RERUN_MLP = False          # True: refit the MLP with 5 seeds (~100 min on CPU, serial; needs classical results)
RERUN_ROBUSTNESS = False   # True: redo the input-noise test (~4 min)
RUN_TESTS = True           # run the pytest suite (~1 min)

## 1. Setup: get the code and data from GitHub and install what is missing

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path("ML-Project-for-ECE").resolve()
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
# Colab/Kaggle already ship numpy, pandas, scipy, scikit-learn, torch and matplotlib; only add what is missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest", "scikit-learn>=1.5"], check=True)
sys.path.insert(0, str(REPO / "python"))
os.environ["PYTHONWARNINGS"] = "ignore"   # silences harmless sklearn GP convergence messages
print("repo at", REPO, "| commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## 2. Does the code pass its own tests in this environment?

In [ ]:
if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-W", "ignore"], capture_output=True, text=True)
    print(r.stdout[-600:] or r.stderr[-600:])

## 3. Stage 1: is the data what we think it is?
`load_dataset` reads the four CSVs and **stops with an error** if anything differs from the verified facts: row counts, NaNs, the reliability flag, the seed, 20 points per curve on one gate grid, and train/test overlap.

In [ ]:
import pandas as pd
from wte2ml.config import load_config, repo_path
from wte2ml.data import curve_matrix, load_dataset, load_layout

cfg = load_config()
ds = load_dataset(cfg)
keys, gate_V, Y = curve_matrix(ds.iv_pooled(), cfg)
layout = load_layout(cfg)
print(f"IV rows: {len(ds.iv_train)} train + {len(ds.iv_test)} test | curves: {len(ds.metrics_train)} + {len(ds.metrics_test)}")
print(f"curve matrix {Y.shape}, gate {gate_V[0]:.2f}-{gate_V[-1]:.2f} V | layouts nested: {layout.is_nested()}")
ds.metrics_pooled().describe().T[["min", "max"]]

## 4. Stage 2: what do we predict, and is the swing extraction trustworthy?
One row per curve, with every target in log10. Swing = steepest PCHIP slope on the falling branch, which ends at the curve minimum. The figure shows 12 sample curves; the dashed line is the steepest tangent.

In [ ]:
from IPython.display import Image, display
from wte2ml.targets import curve_targets
import make_figures

curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
fig_dir = repo_path(cfg, "figures_dir"); fig_dir.mkdir(exist_ok=True)
import matplotlib.pyplot as plt; plt.switch_backend("Agg")
display(Image(str(make_figures.figure_swing_check(cfg, fig_dir))))
curves[["log10_Ion_Ioff", "log10_Ion_Ioff_min", "swing_mV_per_dec", "swing_raw_mV_per_dec", "log10_P_dyn"]].describe().round(3)

## 5. Stage 2: how does the on/off ratio depend on temperature and vacancy?
Rows are temperature in K, columns are edge vacancy in %, and values are the supplied log10 Ion/Ioff.

In [ ]:
curves.pivot(index="T_K", columns="vac_pct", values="log10_Ion_Ioff").iloc[::6].round(1)

## 6. Stage 3: which curves are used for training and which for testing?
There are four protocols. Splits are by whole curve, and hyperparameters are tuned only by blocked CV inside the training pool, with a 1-level embargo.

In [ ]:
import run_experiments as R
R.step_splits(cfg, repo_path(cfg, "results_dir"))
split_summary = pd.read_csv("results/split_summary.csv")
split_summary[split_summary.protocol == split_summary.split][["protocol", "train_curves", "test_curves", "train_T_K", "test_T_K", "test_vac_pct"]]

## 7. Stage 4: do classical models beat a naive guess? (MAE in decades, log10 I)

In [ ]:
out = repo_path(cfg, "results_dir")
if RERUN_CLASSICAL:
    R.step_classical(cfg, out)
classical = pd.read_csv(out / "metrics_classical.csv")
sel = classical[(classical.feature == "vac_pct")]
sel[sel.target == "log10_I"].pivot(index="model", columns="protocol", values="MAE").round(3)

## 8. Stage 4: the same comparison for the on/off ratio, as skill (1 = perfect, 0 = as good as the best naive baseline, < 0 = worse)

In [ ]:
sel[sel.target == "log10_Ion_Ioff"].pivot(index="model", columns="protocol", values="skill").round(2)

## 9. Stage 5: does the neural network beat the simpler models? (mean ± std over 5 seeds)

In [ ]:
if RERUN_MLP:
    R.step_mlp(cfg, out)
mlp = pd.read_csv(out / "mlp_vs_classical.csv")
mlp[["protocol", "target", "mlp_MAE_mean", "mlp_MAE_std", "best_classical_model", "best_classical_MAE", "best_naive_MAE", "mlp_beats_best_classical"]].round(3)

## 10. Stage 6: how many numbers does it take to describe an I-V curve? (PCA)

In [ ]:
R.step_unsupervised(cfg, out)
make_figures.figure_pca_clusters(cfg, fig_dir)
pd.read_csv(out / "pca_explained_variance.csv").head(6).round(4)

## 11. Stage 6: do devices fall into natural groups, and do they match the on/off-ratio classes?
Silhouette: -1 to 1, higher means better separated. ARI: 0 means chance agreement with the Ion/Ioff classes, 1 means identical.

In [ ]:
display(Image(str(fig_dir / "pca_clusters.png")))
pd.read_csv(out / "cluster_scores.csv")[["method", "k", "silhouette", "ari_vs_ion_ioff_classes", "smallest_cluster"]].round(3)

## 12. Stage 6: which vacancy levels end up in which cluster? (k-means, k = 3)

In [ ]:
assign = pd.read_csv(out / "cluster_assignments.csv")
pd.crosstab(assign.vac_pct.round(1), assign.kmeans_k3)

## 13. Stage 7: final model comparison, headline target log10 Ion/Ioff (MAE in decades; rank 1 = best per protocol)

In [ ]:
comparison = pd.read_csv(out / "model_comparison.csv")
comparison[comparison.target == "log10_Ion_Ioff"].pivot(index="model", columns="protocol", values="MAE").round(3)

## 14. Stage 7: is the result the same for the minimum-current definition of Ion/Ioff? (sensitivity)

In [ ]:
comparison[comparison.target == "log10_Ion_Ioff_min"].pivot(index="model", columns="protocol", values="MAE").round(3)

## 15. Stage 7: how much do predictions degrade when temperature and vacancy inputs are noisy?
Noise std as a fraction of one grid step (2 K, 0.1 % vacancy); test MAE of log10 Ion/Ioff, mean over 5 noise draws.

In [ ]:
if RERUN_ROBUSTNESS:
    R.step_robustness(cfg, out)
rob = pd.read_csv(out / "robustness_summary.csv")
display(rob.pivot_table(index=["protocol", "model"], columns="noise_fraction", values="MAE_mean").round(3))
make_figures.report_figures(cfg, fig_dir)   # all Stage 7 figures from results/*.csv
display(Image(str(fig_dir / "robustness_noise.png")))

## 16. Stage 7: where do the models fail? (|error| over the T x vacancy grid, and sample I-V curves with GP bands)

In [ ]:
display(Image(str(fig_dir / "error_map_log10_Ion_Ioff.png")))
display(Image(str(fig_dir / "iv_samples_gp_bands.png")))

## 17. Stage 8: how wide are the error bars? (cluster bootstrap over whole vacancy levels, 95 % CI of the best model's MAE)

In [ ]:
R.step_uncertainty(cfg, out)   # ~1 s, from the saved predictions only
boot = pd.read_csv(out / "bootstrap_ci.csv")
best = comparison[comparison["rank"] == 1][["protocol", "target", "model"]]
best.merge(boot, on=["protocol", "target", "model"])[["target", "protocol", "model", "n_levels", "MAE", "MAE_lo", "MAE_hi"]].round(3)

## 18. Stage 8: can we trust the GP's error bars? (share of test points inside its 68 % and 95 % intervals)

In [ ]:
pd.read_csv(out / "gp_coverage.csv")[["protocol", "target", "coverage_68", "coverage_95", "mean_std"]].round(3)

## 19. Stage 8: would the MLP do better if trained longer? (epoch limit 1,500 vs 8,000; mean MAE over 5 seeds)

In [ ]:
eps = pd.read_csv(out / "epoch_sensitivity.csv")
eps.groupby(["protocol", "target", "max_epochs"]).agg(MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"), seeds_capped=("hit_cap", "sum")).round(3)

## Where to read more
- `docs/REPORT_NOTES.md`: key numbers per target and protocol, failures and limitations, each cited to a results file
- `docs/WALKTHROUGH.md`: the stage-by-stage story with the key numbers
- `docs/ARCHITECTURE.md`: data-flow diagram, modules, glossary, and one section per technique
- `docs/DECISIONS.md`: every design choice with date, stage and rejected alternatives